## Model Training & Evaluation

### Objectives
* Configure Keras `ImageDataGenerator` pipelines with real-time data augmentation on the training set and normalization on validation and test sets.
* Define and compile a custom Convolutional Neural Network (CNN) architecture tailored for binary classification of healthy vs. powdery mildew cherry leaves.
* Implement training callbacks including `EarlyStopping` and `ModelCheckpoint` to optimize performance and prevent overfitting.
* Plot and visualize learning curves for **Model Accuracy** and **Model Loss** across training epochs.
* Evaluate final model performance on the unseen test dataset to derive test loss, test accuracy, and a detailed classification report.
* Compute, visualize, and save a **Confusion Matrix** to analyze false positives and false negatives.
* Export the trained HDF5 model (`.h5`), class index mapping, and evaluation metric artifacts for seamless integration into the Streamlit dashboard.

### Inputs
* Split dataset directories in `inputs/cherry_leaves/train/`, `inputs/cherry_leaves/validation/`, and `inputs/cherry_leaves/test/`

### Outputs
* `outputs/v1/powdery_mildew_detector_model.h5`
* `outputs/v1/class_indices.pkl`
* `outputs/v1/training_history.pkl`
* `outputs/v1/evaluation.pkl`
* `outputs/v1/model_training_history.png`
* `outputs/v1/confusion_matrix.png`

In [3]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

# 1. Resolve Windows DLL Loading and Mute Logs
if sys.platform == "win32":
    os.add_dll_directory(r"C:\Windows\System32")

os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import tensorflow as tf
import keras

# Modern TensorFlow 2.16+ / Keras 3 Utilities
from keras.utils import image_dataset_from_directory
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout, Rescaling, RandomFlip, RandomRotation, RandomZoom
from keras.callbacks import EarlyStopping, ModelCheckpoint

from sklearn.metrics import classification_report, confusion_matrix

# 2. Resolve Project Root Directory
CURRENT_DIR = os.getcwd()
PROJECT_DIR = os.path.dirname(CURRENT_DIR) if os.path.basename(CURRENT_DIR) == 'jupyter_notebooks' else CURRENT_DIR

# 3. Set Input and Output Directories
INPUTS_DIR = os.path.join(PROJECT_DIR, 'inputs', 'cherry_leaves')
TRAIN_DIR = os.path.join(INPUTS_DIR, 'train')
VAL_DIR = os.path.join(INPUTS_DIR, 'validation')
TEST_DIR = os.path.join(INPUTS_DIR, 'test')

OUTPUTS_DIR = os.path.join(PROJECT_DIR, 'outputs', 'v1')
os.makedirs(OUTPUTS_DIR, exist_ok=True)

# Hyperparameters
IMAGE_WIDTH, IMAGE_HEIGHT = 256, 256
IMAGE_SHAPE = (IMAGE_WIDTH, IMAGE_HEIGHT, 3)
BATCH_SIZE = 32
EPOCHS = 25

print(f"TensorFlow Version: {tf.__version__}")
print(f"Keras Version: {keras.__version__}")
print(f"Train Directory: {TRAIN_DIR}")
print(f"Outputs Directory: {OUTPUTS_DIR}")

TensorFlow Version: 2.19.1
Keras Version: 3.15.1
Train Directory: d:\books\code\code-institut\PROJECTS\cherry tree leaves\inputs\cherry_leaves\train
Outputs Directory: d:\books\code\code-institut\PROJECTS\cherry tree leaves\outputs\v1


In [5]:
# 1. Load Training Dataset
train_ds = image_dataset_from_directory(
    TRAIN_DIR,
    labels='inferred',
    label_mode='binary',
    batch_size=BATCH_SIZE,
    image_size=(IMAGE_WIDTH, IMAGE_HEIGHT),
    shuffle=True
)

# 2. Load Validation Dataset
val_ds = image_dataset_from_directory(
    VAL_DIR,
    labels='inferred',
    label_mode='binary',
    batch_size=BATCH_SIZE,
    image_size=(IMAGE_WIDTH, IMAGE_HEIGHT),
    shuffle=False
)

# 3. Load Test Dataset
test_ds = image_dataset_from_directory(
    TEST_DIR,
    labels='inferred',
    label_mode='binary',
    batch_size=BATCH_SIZE,
    image_size=(IMAGE_WIDTH, IMAGE_HEIGHT),
    shuffle=False
)

# 4. Save Class Mappings for Streamlit Deployment
class_names = train_ds.class_names
class_indices = {name: idx for idx, name in enumerate(class_names)}
print(f"\nExtracted Class Indices: {class_indices}")

with open(os.path.join(OUTPUTS_DIR, 'class_indices.pkl'), 'wb') as f:
    pickle.dump(class_indices, f)

# 5. Optimize Dataset Pipelines for I/O Performance
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)

Found 2944 files belonging to 2 classes.
Found 420 files belonging to 2 classes.
Found 844 files belonging to 2 classes.

Extracted Class Indices: {'healthy': 0, 'powdery_mildew': 1}


In [6]:
def create_cnn_model(input_shape):
    """
    Builds a CNN model with built-in rescaling and augmentation layers.
    """
    model = Sequential([
        # Input Layer
        tf.keras.Input(shape=input_shape),

        # Rescaling Layer (Replaces rescale=1./255 from ImageDataGenerator)
        Rescaling(1.0 / 255.0),

        # Data Augmentation Block (Executed only during training)
        RandomFlip("horizontal"),
        RandomRotation(0.1),
        RandomZoom(0.1),

        # First Convolutional Block
        Conv2D(32, (3, 3), activation='relu'),
        MaxPooling2D((2, 2)),

        # Second Convolutional Block
        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D((2, 2)),

        # Third Convolutional Block
        Conv2D(128, (3, 3), activation='relu'),
        MaxPooling2D((2, 2)),

        # Fully Connected Classifier
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    
    return model

model = create_cnn_model(IMAGE_SHAPE)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_flip (RandomFlip)        │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation                 │ (None, 256, 256, 3)    │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_zoom (RandomZoom)        │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 254, 254, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 127, 127, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 125, 125, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 62, 62, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 60, 60, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 30, 30, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 115200)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    14,745,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,839,105 (56.61 MB)

 Trainable params: 14,839,105 (56.61 MB)

 Non-trainable params: 0 (0.00 B)

In [8]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Keras 3 supports .keras or .h5 format
model_save_path = os.path.join(OUTPUTS_DIR, 'powdery_mildew_detector_model.h5')

callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        filepath=model_save_path,
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    )
]

print("Model compiled and callbacks configured.")

Model compiled and callbacks configured.
